# MVGC2 reproducible tutorial — Python parity walk-through

This notebook follows the canonical MVGC2 analysis pipeline using the `complexbox.mvgc` Python port. Its stages correspond to the original MATLAB demo `mvgc_demo_var.m` (Barnett & Seth, *J. Neurosci. Methods*, 2014). Deterministic fixtures protect selected conversion, GC, spectral-resolution, and statistics paths; see `docs/mapping.md` for the precise validation boundary.

## Outline

1. Generate a stationary random VAR(p) ground-truth model.
2. Simulate multi-trial time-series data.
3. Select VAR model order (AIC / BIC / HQC).
4. Fit the VAR (LWR and OLS).
5. Diagnose the fit (consistency, residual whiteness).
6. Compute pairwise-conditional Granger causality (time-domain).
7. Compute spectral (frequency-domain) GC.
8. Significance testing.
9. State-space pathway: convert VAR → innovations SS → same GC results.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import trapezoid

from complexbox import mvgc

rng = np.random.default_rng(20260516)
np.set_printoptions(precision=4, suppress=True)

## 1. Random ground-truth VAR model

We construct a 5-variable VAR(3) with target spectral radius 0.9 (close to but inside the unit circle — typical of neural / climate time series). The residuals covariance is a random positive-definite matrix sampled via the *onion* method.

In [ ]:
n_vars, p_true = 5, 3
A_true = mvgc.var_rand(n_vars, p_true, rho=0.9, rng=rng)
V_true = mvgc.corr_rand(n_vars, rng=rng)

print(f"spectral radius = {mvgc.specnorm(A_true):.6f}")
print("residuals covariance V_true:")
print(V_true)

## 2. Simulate data

Run the VAR forward to produce one trial of 10,000 samples. The simulator automatically truncates the start-up transient based on the spectral radius (the same heuristic MVGC2 uses).

In [ ]:
m = 10_000  # observations per trial
N = 1  # trials
X, E = mvgc.var_to_tsdata(A_true, V_true, m=m, N=N, rng=rng)
print("X shape:", X.shape)

fig, axes = plt.subplots(n_vars, 1, figsize=(10, 6), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(X[i, :500], lw=0.7)
    ax.set_ylabel(f"x{i + 1}")
axes[-1].set_xlabel("time sample")
fig.suptitle("first 500 samples of simulated VAR data")
fig.tight_layout()

## 3. Model-order selection

Try VAR orders 1...10 and pick the one minimising each information criterion. AIC tends to overshoot; BIC undershoots; HQC sits in between.

In [ ]:
mo = mvgc.tsdata_to_varmo(X, pmax=10, regmode="LWR")
print(f"AIC -> p = {mo.p_aic}")
print(f"BIC -> p = {mo.p_bic}")
print(f"HQC -> p = {mo.p_hqc}")
print(f"truth: p = {p_true}")

fig, ax = plt.subplots(figsize=(8, 4))
p_axis = np.arange(1, 11)
ax.plot(p_axis, mo.aic, "o-", label="AIC")
ax.plot(p_axis, mo.bic, "s-", label="BIC")
ax.plot(p_axis, mo.hqc, "^-", label="HQC")
ax.axvline(p_true, ls="--", c="k", alpha=0.5, label=f"true order p={p_true}")
ax.set_xlabel("VAR order p")
ax.set_ylabel("information criterion (per-observation)")
ax.legend()
ax.set_title("VAR model-order selection")

## 4. Fit the VAR using LWR and OLS

Both algorithms are exposed; LWR (Morf's lattice-whitening recursion) is stable by construction, OLS via QR is valid for unstable processes.

In [ ]:
p = mo.p_hqc
fit_lwr = mvgc.tsdata_to_var(X, p=p, regmode="LWR")
fit_ols = mvgc.tsdata_to_var(X, p=p, regmode="OLS")

print(f"fit p = {p}")
print(f"rho(A_LWR) = {mvgc.specnorm(fit_lwr.A):.6f}")
print(f"rho(A_OLS) = {mvgc.specnorm(fit_ols.A):.6f}")
if p >= p_true:
    err = np.max(np.abs(fit_lwr.A[..., :p_true] - A_true))
    print(f"max |A_LWR - A_true| over first {p_true} lags = {err:.4e}")

## 5. Diagnostics: consistency and whiteness

The Ding-Bressler consistency statistic should be > 0.8 for a good fit. The Durbin-Watson test (per variable) checks that residuals are temporally uncorrelated.

In [ ]:
cons = mvgc.consistency(X, fit_lwr.E)
dw, pval = mvgc.whiteness(X, fit_lwr.E)
print(f"consistency = {cons:.4f}")
print(f"Durbin-Watson statistics per variable: {dw}")
print(f"whiteness p-values:                   {pval}")

## 6. Time-domain pairwise-conditional Granger causality

`F[i, j]` is the causality **from `j` to `i`**, conditioning out all other variables.

In [ ]:
F_true = mvgc.var_to_pwcgc(A_true, V_true)
F_hat = mvgc.var_to_pwcgc(fit_lwr.A, fit_lwr.V)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, mat, title in zip(axes, (F_true, F_hat), ("ground truth", "LWR estimate")):
    im = ax.imshow(np.where(np.isnan(mat), 0, mat), cmap="viridis", vmin=0)
    plt.colorbar(im, ax=ax)
    ax.set_title(f"pairwise-conditional GC — {title}")
    ax.set_xlabel("source j")
    ax.set_ylabel("target i")
fig.tight_layout()

## 7. Spectral (frequency-domain) Granger causality

Integrated over the full spectrum, the spectral GC must agree with the time-domain estimate. Below we compare for one (target, source) pair.

In [ ]:
fres = 256
F_spec = mvgc.var_to_spwcgc(fit_lwr.A, fit_lwr.V, fres=fres)
omega = np.linspace(0, np.pi, fres + 1)

fig, ax = plt.subplots(figsize=(10, 4))
for i in range(n_vars):
    for j in range(n_vars):
        if i == j:
            continue
        ax.plot(omega, F_spec[i, j, :], lw=0.8, label=f"{j + 1} -> {i + 1}")
ax.set_xlabel("frequency (rad)")
ax.set_ylabel("spectral GC")
ax.set_title("pairwise-conditional spectral GC")
ax.legend(ncol=2, fontsize=8)

# Integrated spectral GC matches the time-domain matrix
F_int = trapezoid(np.nan_to_num(F_spec, nan=0), omega, axis=2) / np.pi
print(
    "max |integrated spectral GC - time-domain GC| =",
    np.nanmax(np.abs(F_int - np.nan_to_num(F_hat, nan=0))),
)

## 8. Statistical significance testing

Under the null of no causality, the GC statistic follows an F (or LR chi^2) distribution. We compute p-values for every off-diagonal entry and apply Benjamini-Hochberg FDR control.

In [ ]:
pvals = np.full_like(F_hat, np.nan)
for i in range(n_vars):
    for j in range(n_vars):
        if i == j:
            continue
        pvals[i, j] = mvgc.mvgc_pval(
            F_hat[i, j],
            "F",
            nx=1,
            ny=1,
            nz=n_vars - 2,
            p=p,
            m=m,
            N=N,
        )

sig = mvgc.significance(pvals, alpha=0.05, method="FDR")
print("FDR-significant connections (i <- j):")
print(sig.astype(int))

## 9. State-space pathway — same GC, different route

Convert the fitted VAR to innovations-form SS via the companion matrix; the GC values must match the VAR-based computation to machine precision.

In [ ]:
A_ss, C, K, _ = mvgc.var_to_ss(fit_lwr.A, fit_lwr.V)
F_ss = mvgc.ss_to_pwcgc(A_ss, C, K, fit_lwr.V)

diff = np.nanmax(np.abs(F_hat - F_ss))
print(f"max |F_VAR - F_SS| = {diff:.4e}  (should be machine precision)")
assert diff < 1e-8

## Validation against MATLAB

If you have MATLAB + MVGC2 installed, regenerate the reference `.mat` fixtures via `tools/matlab_fixtures/generate_all_fixtures.m`, then run `pytest -m fixture` to assert that the Python results match the MATLAB ones to 1e-10 or tighter.

## Further reading

- L. Barnett and A. K. Seth, "The MVGC Multivariate Granger Causality Toolbox", *J. Neurosci. Methods* 223 (2014).
- L. Barnett and A. K. Seth, "Granger causality for state-space models", *Phys. Rev. E* 91, 040101(R) (2015).